In [1]:
import pandas as pd
import numpy as np

In [2]:
from pulp import *

In [3]:
training_ds = pd.read_csv(
    "data_flat/training_dataset.csv"
)

travel_times = pd.read_csv(
    "data_flat/travel_times.csv"
)

In [ ]:
travel_dict = {}

for _, row in travel_times.iterrows():

    travel_dict[
        (
            row["route_id"],
            row["from_stop"],
            row["to_stop"]
        )
    ] = row["travel_time_seconds"]

In [ ]:
#cost function
BIG_M = 1e6


def cost(
        route_id,
        from_stop,
        to_stop
):

    if from_stop == to_stop:
        return 0

    return travel_dict.get(
        (
            route_id,
            from_stop,
            to_stop
        ),
        BIG_M
    )

In [ ]:
route_id = training_ds["route_id"].iloc[0]

route_df = training_ds[
    training_ds["route_id"] == route_id
].copy()

In [ ]:
route_df.shape

In [ ]:
station_stop = route_df.loc[
    route_df["type"] == "Station",
    "stop_id"
].iloc[0]

print(station_stop)

In [ ]:
actual_stops = (
    route_df
    .sort_values("actual_sequence")
    ["stop_id"]
    .tolist()
)

actual_stops

In [ ]:
def edge_accuracy_stops(
        pred,
        true
):

    pred_edges = set(
        zip(pred[:-1], pred[1:])
    )

    true_edges = set(
        zip(true[:-1], true[1:])
    )

    return (
        len(pred_edges & true_edges)
        /
        len(true_edges)
    )

In [ ]:
def zone_stops(
        route_df,
        zone_id
):

    return route_df.loc[
        (
            route_df["zone_id"] == zone_id
        )
        &
        (
            route_df["type"] != "Station"
        ),
        "stop_id"
    ].tolist()

In [ ]:
#test
zone_example = route_df["zone_id"].dropna().iloc[0]

zone_stops(
    route_df,
    zone_example
)


In [ ]:
#anchor zona successiva come in maragno
def next_zone_anchor(
        route_df,
        next_zone
):

    next_zone_df = route_df[
        route_df["zone_id"] == next_zone
    ].copy()

    if len(next_zone_df) == 0:
        return None

    center_lat = next_zone_df["lat"].mean()

    center_lng = next_zone_df["lng"].mean()

    next_zone_df["dist_to_center"] = np.sqrt(
        (next_zone_df["lat"] - center_lat) ** 2
        +
        (next_zone_df["lng"] - center_lng) ** 2
    )

    anchor = next_zone_df.loc[
        next_zone_df["dist_to_center"].idxmin(),
        "stop_id"
    ]

    return anchor

In [ ]:
zones = (
    route_df["zone_id"]
    .dropna()
    .unique()
)

next_zone = zones[1]

next_zone_anchor(
    route_df,
    next_zone
)
